# Pipeline satu lagu di iPhone: palette6, CPU only

Mengukur PCM 24 kHz → log-mel → MusicFM + MuQ (stream 30/120 detik) → head → probabilitas delapan kelas → pooling sesuai section Apple yang sudah tersedia. Membandingkan prediksi dengan hasil Mac pada lagu yang sama.

Window terakhir memakai panjang asli, **tanpa zero padding**. Semua model memakai CPU agar pengujian ini tidak memanggil jalur dynamic-shape GPU yang sebelumnya assert. Tahap ini memverifikasi pipeline lengkap dahulu; bukan benchmark jalur GPU optimal.

Audio MP3 didekode dan di-resample di Mac; aplikasi uji memakai PCM yang sama. Deteksi boundary MusicUnderstanding dan decoder audio iPhone belum diuji. Proyek terpisah dari SongMap. Sediakan beberapa GiB disk untuk kompilasi; aplikasi menyimpan keenam model dalam working set CPU.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, subprocess, sys, hashlib
assert Path('finetuning/config.json').is_file()
SONGMAP_ROOT=Path('/Users/danendra/Documents/ADA26/C6/SongMap')
reports=sorted(Path('outputs.local').glob('muq-compression-*/palette6-all-validation.json'),key=lambda p:p.stat().st_mtime,reverse=True)
assert reports,'Validation MuQ palette6 belum ditemukan'
validation=reports[0]
v=json.loads(validation.read_text())
combination=json.loads(Path(v['baseline_combination']).read_text())
pipeline=json.loads(Path(combination['encoder_pipeline_report']).read_text())
expected=pipeline['musicfm_override']['report_sha256']
proofs=[p for p in Path('outputs.local').glob('musicfm-compression-*/palette6/compression_report.json')
        if hashlib.sha256(p.read_bytes()).hexdigest()==expected]
assert len(proofs)==1,'Isi musicfm_proof secara manual dengan report kompresi MusicFM yang sesuai'
musicfm_proof=proofs[0]
output=Path('outputs.local')/('pipeline-iphone-'+datetime.now().strftime('%Y%m%d-%H%M%S'))
# Kosong = lagu validation pertama. Isi track_id untuk lagu lain setelah pengujian pertama.
TRACK_ID=''
print('MuQ validation:',validation)
print('MusicFM proof:',musicfm_proof)
print('Output:',output)


In [ ]:
command=[sys.executable,'-u','-m','deployment.prepare_pipeline_iphone',
         '--songmap-root',str(SONGMAP_ROOT),'--muq-validation',str(validation),
         '--musicfm-compression-report',str(musicfm_proof),'--output-dir',str(output)]
if TRACK_ID:
    command+=['--track-id',TRACK_ID]
log=output.with_suffix('.prepare.log')
log.parent.mkdir(parents=True,exist_ok=True)
with log.open('w',encoding='utf-8') as handle:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='');handle.write(line);handle.flush()
        status=process.wait()
    except KeyboardInterrupt:
        process.terminate();process.wait();raise
if status:
    raise RuntimeError(f'Persiapan gagal ({status}); lihat {log.resolve()}')
subprocess.run(['open',str(output/'Benchmark.xcodeproj')],check=True)


## Jalankan di iPhone

1. Pilih Signing Team dan target iPhone fisik di Xcode.
2. Jalankan aplikasi, tekan **Run full pipeline**, dan biarkan aplikasi terbuka sampai selesai. Log Xcode mencetak progres setiap head chunk. CPU memerlukan waktu lebih lama daripada benchmark satu encoder dengan mode `all`.
3. Tekan **Share JSON report** lalu kirim `pipeline-benchmark-*.json`.

Periksa `section_agreement_with_mac`, `max_section_probability_error`, `all_predictions_finite`, `low_coverage_sections`, serta waktu tiap komponen. Snapshot memori bukan peak memory. Data model dan PCM disertakan dalam aplikasi benchmark; jangan membagikan bundle secara publik.

Belum otomatis disetujui untuk integrasi. Setelah hasil lengkap cocok, optimasi berikutnya dapat memakai model statis untuk window penuh dan CPU untuk sisa window, dengan pengujian ulang pengaruh perpindahan jalur.